# Module 2: Banking Transformations

**Objective**: Learn practical data transformations for banking analytics.

## What You'll Learn
1. Multi-table joins (Customers ↔ Accounts ↔ Transactions)
2. Customer segmentation logic
3. Aggregations and grouping
4. Creating derived columns with business logic

In [1]:
# ── SparkSession: Databricks Connect (remote) / Local fallback ──
from pathlib import Path

try:
    from databricks.connect import DatabricksSession # pyright: ignore[reportMissingImports]
    spark = DatabricksSession.builder.serverless().getOrCreate()
    MODE = 'databricks'
    print(f"✅ Databricks Connect | Spark {spark.version}")
except Exception:
    from pyspark.sql import SparkSession # pyright: ignore[reportMissingImports]
    spark = SparkSession.builder.appName("Module02").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    MODE = 'local'
    print(f"✅ Local Spark {spark.version} | UI: http://localhost:4040")

DATA_RAW = Path("../data/raw")  # local CSV fallback path
print(f"Mode: {MODE}")

✅ Databricks Connect | Spark 4.1.0
Mode: databricks


In [3]:
# ── Load data (Delta format from Bronze layer) ──
S3_BRONZE = "s3a://sparkling-data-test/migration/bronze"
DATA_BRONZE = Path("../data/bronze")

dim_branch_df = spark.read.format("delta").load(f"{S3_BRONZE}/dim_branch")
dim_date_df = spark.read.format("delta").load(f"{S3_BRONZE}/dim_date")

dim_customer_df = spark.read.format("delta").load(f"{S3_BRONZE}/dim_customer")
dim_account_df = spark.read.format("delta").load(f"{S3_BRONZE}/dim_account")
dim_account_type_df = spark.read.format("delta").load(f"{S3_BRONZE}/dim_account_type")

fact_daily_balance_df = spark.read.format("delta").load(f"{S3_BRONZE}/fact_daily_balance")
fact_transaction_df = spark.read.format("delta").load(f"{S3_BRONZE}/fact_transaction")

# Map new tables to existing notebook variables for compatibility
customers_df = dim_customer_df
accounts_df = dim_account_df
transactions_df = fact_transaction_df
branches_df = dim_branch_df

print(f"Loaded: {customers_df.count()} customers, {accounts_df.count()} accounts, {transactions_df.count()} transactions, {branches_df.count()} branches, {fact_daily_balance_df.count()} daily balances, {fact_transaction_df.count()} transactions, {dim_account_type_df.count()} account types, {dim_date_df.count()} dates" )


Loaded: 12235 customers, 20269 accounts, 3711534 transactions, 100 branches, 2960000 daily balances, 3711534 transactions, 6 account types, 2557 dates


In [11]:
S3_RAW = 's3://sparkling-data-test/data/raw/'
accounts_df = spark.read.parquet(f"{S3_RAW}/accounts")

In [4]:
import sys, os, importlib
sys.path.insert(0, os.path.abspath(".."))
from src.schema_utils import schema_class, generate_schema_source

# ── Build runtime schema classes (tab-complete & IntelliSense) ──
BranchCols       = schema_class(branches_df,      "BranchCols")
TransactionsCols  = schema_class(transactions_df,  "TransactionsCols")
CustomerCols     = schema_class(customers_df,     "CustomerCols")
AccountCols      = schema_class(accounts_df,      "AccountCols")
AccountTypeCols = schema_class(dim_account_type_df, 'AccountTypeCols')
DailyBalancesCols = schema_class(fact_daily_balance_df, 'DailyBalancesCols')

# Verify ColumnDescriptor is a str subclass (should be True)

## Multi-Table Joins

In [8]:
# Join customers with accounts
# customer_accounts = customers_df.join(accounts_df, AccountCols.ACCOUNT_ID, "inner")
customer_accounts = customers_df.join(accounts_df, CustomerCols.CUSTOMER_ID, "inner")
customer_accounts.select(
    CustomerCols.CUSTOMER_ID, 
    CustomerCols.FULL_NAME, 
    CustomerCols.SEGMENT, 
    AccountCols.ACCOUNT_ID, 
    AccountCols.BALANCE
).show(5)

+-----------+--------------+---------+----------+-----------------+
|customer_id|     full_name|  segment|account_id|          balance|
+-----------+--------------+---------+----------+-----------------+
| CUST000004|Pham Hung Uyen|     Mass|ACCT000004|       1225216.03|
| CUST000015|Nguyen Van Chi| Affluent|ACCT000008|    1.297098811E7|
| CUST000012|  Le Van Quang|     Mass|ACCT000012|    1.984677702E7|
| CUST000013| Bui Van Giang|      VIP|ACCT000016|4.278053757601E10|
| CUST000017|   Dang Lan Em|Corporate|ACCT000020|4.064728499143E10|
+-----------+--------------+---------+----------+-----------------+
only showing top 5 rows


## Customer Aggregations

In [4]:
from pyspark.sql.functions import count, sum, avg, round as spark_round, col, when, date_format, to_date

In [5]:
# Customer summary
customer_summary = accounts_df.groupBy("customer_id").agg(
    count("account_id").alias("account_count"),
    spark_round(sum("balance"), 2).alias("total_balance"),
    spark_round(avg("balance"), 2).alias("avg_balance")
)
customer_summary.orderBy(col("total_balance").desc()).show(10)

+-----------+-------------+------------------+-----------------+
|customer_id|account_count|     total_balance|      avg_balance|
+-----------+-------------+------------------+-----------------+
| CUST004750|            3|1.2462703053777E11|4.154234351259E10|
| CUST000621|            3|1.1948662840498E11|3.982887613499E10|
| CUST008150|            3| 1.153007508303E11| 3.84335836101E10|
| CUST002603|            3|1.1480851364991E11|3.826950454997E10|
| CUST009481|            3|1.0858819564595E11|3.619606521532E10|
| CUST001775|            3|1.0082519449151E11|3.360839816384E10|
| CUST005044|            3|1.0004758086192E11|3.334919362064E10|
| CUST007747|            3| 9.955017590227E10|3.318339196742E10|
| CUST004892|            2| 9.815940347914E10|4.907970173957E10|
| CUST004656|            2|  9.71458185112E10| 4.85729092556E10|
+-----------+-------------+------------------+-----------------+
only showing top 10 rows


## Customer Segmentation Logic

In [6]:
# Re-segment based on balance
customer_profile = customers_df.join(customer_summary, "customer_id", "left").fillna({"total_balance": 0})
customer_resegment = customer_profile.withColumn(
    "calculated_segment",
    when(col("total_balance") >= 5_000_000_000, "UHNW")
    .when(col("total_balance") >= 500_000_000, "HNW")
    .when(col("total_balance") >= 100_000_000, "Affluent")
    .when(col("total_balance") >= 20_000_000, "Mass Affluent")
    .otherwise("Mass")
)
customer_resegment.select("name", "segment", "calculated_segment", "total_balance").show(10)

+--------------+-------------+------------------+---------------+
|          name|      segment|calculated_segment|  total_balance|
+--------------+-------------+------------------+---------------+
|Customer 06251|         Mass|              Mass|  1.063612138E7|
|Customer 06252|Mass Affluent|          Affluent| 1.3522805773E8|
|Customer 06253|         Mass|              Mass|  1.811233082E7|
|Customer 06254|Mass Affluent|          Affluent| 1.0016471999E8|
|Customer 06255|          HNW|               HNW|2.99095530645E9|
|Customer 06256|         Mass|              Mass|  1.514097044E7|
|Customer 06257|Mass Affluent|          Affluent| 1.3316701797E8|
|Customer 06258|Mass Affluent|     Mass Affluent|  5.713734625E7|
|Customer 06259|Mass Affluent|     Mass Affluent|  8.510116382E7|
|Customer 06260|         Mass|     Mass Affluent|  2.592709984E7|
+--------------+-------------+------------------+---------------+
only showing top 10 rows


## Transaction Analysis

In [7]:
# Monthly by channel
txn_parsed = transactions_df.withColumn("txn_month", date_format(to_date(col("txn_datetime")), "yyyy-MM"))
monthly_by_channel = txn_parsed.groupBy("txn_month", "channel").agg(count("*").alias("count"), sum("amount").alias("total"))
monthly_by_channel.orderBy("txn_month").show(15)

+---------+----------------+-----+--------------------+
|txn_month|         channel|count|               total|
+---------+----------------+-----+--------------------+
|  2025-01|             API|70655|3.917239225343873E12|
|  2025-01|Internet Banking|71024| 3.96100423173662E12|
|  2025-01|          Branch|70992|3.933845600689784...|
|  2025-01|             POS|70664|3.928462615533408E12|
|  2025-01|      Mobile App|70819|3.924665539435593E12|
|  2025-01|             ATM|70404|3.917175605840881...|
|  2025-02|Internet Banking|64150|3.530528354875180...|
|  2025-02|          Branch|64031|3.567835897754383...|
|  2025-02|             POS|63632|3.555105351798108E12|
|  2025-02|             ATM|64272|3.591315417100188E12|
|  2025-02|      Mobile App|63813|3.537048210491230...|
|  2025-02|             API|63872|3.550060993394684E12|
|  2025-03|             POS|70640|3.944063533545708...|
|  2025-03|Internet Banking|70740| 3.91305075831661E12|
|  2025-03|          Branch|70976|3.919087300167

In [8]:
# Transaction categorization
txn_cat = transactions_df.withColumn("size",
    when(col("amount") >= 100_000_000, "Large").when(col("amount") >= 10_000_000, "Medium").otherwise("Small")
)
txn_cat.groupBy("size").count().show()

+------+-------+
|  size|  count|
+------+-------+
| Large| 398369|
|Medium|3397519|
| Small|1204112|
+------+-------+



## Practice Exercises
1. Find top 10 branches by transaction volume
2. Calculate deposit-to-withdrawal ratio per customer
3. Find customers with no transactions (left_anti join)

In [ ]:
from pyspark.sql import functions as F

top_10_branches = (
    transactions_df
    .join(accounts_df, TransactionsCols.ACCOUNT_ID)          # shared col name → no duplicate
    .join(branches_df, AccountCols.BRANCH_ID)               # shared col name → no duplicate
    .groupBy(BranchCols.BRANCH_ID, BranchCols.BRANCH_NAME)
    .agg(F.count("*").alias("txn_count"))
    .select(BranchCols.BRANCH_ID, BranchCols.BRANCH_NAME, "txn_count")
    .orderBy(F.col("txn_count").desc())
    # .limit(10)
)

top_10_branches.show()

+---------+--------------------+---------+
|branch_id|         branch_name|txn_count|
+---------+--------------------+---------+
| BR000036|Ho Chi Minh Branc...|   116587|
| BR000016|  Can Tho Branch 016|    91784|
| BR000040|Binh Duong Branch...|    91331|
| BR000008|Hai Phong Branch 008|    87827|
| BR000024| Dong Nai Branch 024|    83921|
| BR000079|    Hanoi Branch 079|    81695|
| BR000030|Binh Duong Branch...|    81628|
| BR000026|Ho Chi Minh Branc...|    80872|
| BR000046|Quang Ninh Branch...|    79998|
| BR000012|Binh Duong Branch...|    79069|
| BR000084|Hai Phong Branch 084|    78030|
| BR000055| Dong Nai Branch 055|    76594|
| BR000058|Quang Ninh Branch...|    75302|
| BR000022|Quang Ninh Branch...|    72529|
| BR000009| Dong Nai Branch 009|    70933|
| BR000054| Dong Nai Branch 054|    68188|
| BR000004|Hai Phong Branch 004|    67807|
| BR000061|    Hanoi Branch 061|    67533|
| BR000074|  Can Tho Branch 074|    67273|
| BR000082|  Da Nang Branch 082|    67217|
+---------+

In [12]:
generate_schema_source(transactions_df)

class Cols:
    """Auto-generated column constants. 17 columns."""

    TXN_KEY: str = "txn_key"  # Spark: Long | Python: int | nullable
    TXN_ID: str = "txn_id"  # Spark: String | Python: str | nullable
    CUSTOMER_ID: str = "customer_id"  # Spark: String | Python: str | nullable
    BRANCH_ID: str = "branch_id"  # Spark: String | Python: str | nullable
    ACCOUNT_TYPE_CODE: str = "account_type_code"  # Spark: String | Python: str | nullable
    TXN_DATETIME: str = "txn_datetime"  # Spark: Timestamp | Python: datetime.datetime | nullable
    TXN_TYPE: str = "txn_type"  # Spark: String | Python: str | nullable
    AMOUNT: str = "amount"  # Spark: Decimal(18,2) | Python: decimal.Decimal | nullable
    CURRENCY: str = "currency"  # Spark: String | Python: str | nullable
    CHANNEL: str = "channel"  # Spark: String | Python: str | nullable
    STATUS: str = "status"  # Spark: String | Python: str | nullable
    DESCRIPTION: str = "description"  # Spark: String | Python: str | nullabl

'class Cols:\n    """Auto-generated column constants. 17 columns."""\n\n    TXN_KEY: str = "txn_key"  # Spark: Long | Python: int | nullable\n    TXN_ID: str = "txn_id"  # Spark: String | Python: str | nullable\n    CUSTOMER_ID: str = "customer_id"  # Spark: String | Python: str | nullable\n    BRANCH_ID: str = "branch_id"  # Spark: String | Python: str | nullable\n    ACCOUNT_TYPE_CODE: str = "account_type_code"  # Spark: String | Python: str | nullable\n    TXN_DATETIME: str = "txn_datetime"  # Spark: Timestamp | Python: datetime.datetime | nullable\n    TXN_TYPE: str = "txn_type"  # Spark: String | Python: str | nullable\n    AMOUNT: str = "amount"  # Spark: Decimal(18,2) | Python: decimal.Decimal | nullable\n    CURRENCY: str = "currency"  # Spark: String | Python: str | nullable\n    CHANNEL: str = "channel"  # Spark: String | Python: str | nullable\n    STATUS: str = "status"  # Spark: String | Python: str | nullable\n    DESCRIPTION: str = "description"  # Spark: String | Python

In [13]:
from pyspark.sql.functions import col, sum, when, round as spark_round

# Join transactions → accounts to get customer_id
txn_with_customer = transactions_df.join(customers_df, CustomerCols.CUSTOMER_ID, "inner") \
    .select(AccountCols.CUSTOMER_ID, TransactionsCols.TXN_TYPE, TransactionsCols.AMOUNT)

# Deposit-to-withdrawal ratio per customer
deposit_withdrawal_ratio = txn_with_customer.groupBy(AccountCols.CUSTOMER_ID).agg(
    spark_round(sum(when(col(TransactionsCols.TXN_TYPE) == "Deposit", col(TransactionsCols.AMOUNT)).otherwise(0)), 2).alias("total_deposits"),
    spark_round(sum(when(col(TransactionsCols.TXN_TYPE) == "Withdrawal", col(TransactionsCols.AMOUNT)).otherwise(0)), 2).alias("total_withdrawals"),
).withColumn(
    "deposit_to_withdrawal_ratio",
    spark_round(
        when(col("total_withdrawals") > 0, col("total_deposits") / col("total_withdrawals"))
        .otherwise(None),
        4
    )
)

deposit_withdrawal_ratio.orderBy(col("deposit_to_withdrawal_ratio").desc()).show()


+-----------+---------------+-----------------+---------------------------+
|customer_id| total_deposits|total_withdrawals|deposit_to_withdrawal_ratio|
+-----------+---------------+-----------------+---------------------------+
| CUST008886|122624482972.71|   31982791851.12|                     3.8341|
| CUST009895| 16865200300.86|    6816408866.86|                     2.4742|
| CUST004409| 14174040543.99|    5924278847.28|                     2.3925|
| CUST007136| 58375144031.04|   25026015049.96|                     2.3326|
| CUST000025| 22219561928.46|    9953857854.42|                     2.2323|
| CUST006964| 45651800973.66|   22079488204.70|                     2.0676|
| CUST006047|  7237943445.98|    3688981448.70|                     1.9620|
| CUST009605|  2677923218.72|    1440845856.63|                     1.8586|
| CUST000355| 56520470974.94|   30450928060.78|                     1.8561|
| CUST007504|  2332233845.59|    1261131941.08|                     1.8493|
| CUST007671

In [17]:
# First, get all customer_ids that have at least one transaction
# (transactions link to customers through accounts)
accounts_with_txns = accounts_df\
    .join(customers_df, AccountCols.CUSTOMER_ID) \
    .join(transactions_df, TransactionsCols.CUSTOMER_ID) \
    .select(AccountCols.CUSTOMER_ID).distinct()
# left_anti: keep only customers NOT found in accounts_with_txns
customers_no_txns = customers_df.join(accounts_with_txns, AccountCols.CUSTOMER_ID, "left_anti")
print(f"Customers with no transactions: {customers_no_txns.count()}")
# customers_no_txns.select("customer_id", "name", "segment", "registration_date").show(15)

Customers with no transactions: 0


In [ ]:
spark.stop()